# Task 1: Anti-Cheat Pre-Filter (Binary Classification)

Predict `is_cheater` (1 = cheater) so the review team can prioritise suspicious accounts.

| Section | What it does |
|---|---|
| 1 | Load data, basic overview |
| 2 | **Which metric Kaggle really uses**: binary F2 on the cheater class |
| 3 | **Train/test difference**: test rows are far more complete → weight validation to match |
| 4 | Model: regularised LightGBM, 5-fold out-of-fold (OOF) predictions |
| 5 | Choose the decision threshold for binary F2 (honest nested check) |
| 6 | What else was tried (and why it was not kept) |
| 7 | Fit on all labelled rows → predict test → `task1_prediction.csv` |

**Leakage rules:** `player_id` is never a feature; the test set is only used for prediction and for counting
*how many feature values are missing* per row (section 3), never for labels or fitting.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import fbeta_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Kaggle files: put train.csv / test.csv next to this notebook, or keep the team data folder beside the repo
CANDIDATES = [Path('.'), Path('../../cpe342-kaggle-main/01_binary_classification')]
DATA = next(d for d in CANDIDATES if (d / 'train.csv').exists())
SAMPLE = next(p for p in [Path('../sample_submissionMLgroup.csv'), Path('sample_submissionMLgroup.csv')] if p.exists())
TARGET, SEED = 'is_cheater', 42

train = pd.read_csv(DATA / 'train.csv'); test = pd.read_csv(DATA / 'test.csv')
FEATS = [c for c in train.columns if c not in ('player_id', TARGET)]
lab = train[train[TARGET].notna()].reset_index(drop=True)
y = lab[TARGET].astype(int).values
print(f'data from {DATA.resolve()}')
print(f'train {train.shape} (labelled {len(lab):,}, unlabelled {train[TARGET].isna().sum():,}) | test {test.shape} | {len(FEATS)} features')

data from D:\coding\CPE342ML\cpe342-kaggle-main\01_binary_classification
train (104000, 33) (labelled 90,816, unlabelled 13,184) | test (25889, 32) | 31 features


## 1. Overview

In [2]:
print('cheater share (labelled):', round(y.mean(), 3))
miss = pd.DataFrame({'train %': lab[FEATS].isna().mean() * 100, 'test %': test[FEATS].isna().mean() * 100}).round(1)
display(miss.sort_values('train %', ascending=False).head(10))
print('missing cells overall: train %.1f%% | test %.1f%%' % (lab[FEATS].isna().mean().mean() * 100, test[FEATS].isna().mean().mean() * 100))

cheater share (labelled): 0.349


,train %,test %
survival_time_avg,24.3,13.1
weapon_switch_speed,24.2,15.5
avg_session_length_min,22.9,1.6
movement_pattern_score,22.5,8.7
reports_received,22.3,14.9
friend_network_size,22.3,4.7
aiming_smoothness,20.8,2.1
communication_rate,20.5,11.0
team_play_score,19.5,8.1
spray_control_score,16.7,5.6


missing cells overall: train 12.0% | test 8.0%


- 90,816 labelled rows, 34.9% cheaters. 13,184 training rows have no label and are not used for training (pseudo-labelling them was tested in section 6 and did not help).
- Missing values are **not** a clue to the class (rates are the same for cheaters and legit players), but test has far fewer of them.

## 2. Which metric does Kaggle use?
The Kaggle Overview page says *F2 (Macro)*, but the project PDF (p.15) gives only `F2 = 5·P·R / (4P + R)`, and only Task 4 says *Macro*.
A leaderboard test settled it: changing only the Task 1 threshold from the Macro-F2 optimum to the binary-F2 optimum **raised** the total score
(the reverse change had lowered it by about 0.015). So Kaggle scores **binary F2 on the cheater class**:
`fbeta_score(y, pred, beta=2)`. F2 weights recall 4× more than precision → the best threshold is low and many players get flagged.

In [3]:
def f2(y_true, pred, w=None):             # metric used by Kaggle for Task 1
    return fbeta_score(y_true, pred, beta=2, sample_weight=w)
def macro_f2(y_true, pred, w=None):       # what the Overview page suggests (kept for comparison)
    return fbeta_score(y_true, pred, beta=2, average='macro', sample_weight=w)

## 3. Train and test are not the same mix of rows
The five strongest features are often missing in train but almost always present in test.
Rows with all five present are easier to classify, so a plain cross-validation score is **pessimistic** for the leaderboard.
We weight each training row so the validation mix matches the test mix (by number of missing key features).

In [4]:
KEY = ['reports_received', 'crosshair_placement', 'account_age_days', 'friend_network_size', 'level']
n_miss_tr = lab[KEY].isna().sum(axis=1).values
n_miss_te = test[KEY].isna().sum(axis=1).values
mix = pd.DataFrame({'train share': pd.Series(n_miss_tr).value_counts(normalize=True),
                    'test share': pd.Series(n_miss_te).value_counts(normalize=True)}).sort_index().fillna(0).round(3)
mix.index.name = 'key features missing'
display(mix)
W = np.array([(n_miss_te == k).mean() / (n_miss_tr == k).mean() for k in n_miss_tr])   # test-mix weights for validation only

,train share,test share
key features missing,,
0,0.418,0.772
1,0.404,0.211
2,0.150,0.016
3,0.026,0.000
4,0.002,0.000
5,0.000,0.000


77% of test rows have all key features, against 42% of train. All validation scores below use these weights (they are *not* used to train the model; training with them was tested and did not help).

## 4. Model: regularised LightGBM, 5-fold OOF

In [5]:
PARAMS = dict(n_estimators=2000, learning_rate=0.01, num_leaves=15, min_child_samples=100,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.6, verbose=-1, random_state=0)
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
oof = np.zeros(len(y))
for tr, va in folds:
    m = lgb.LGBMClassifier(**PARAMS).fit(lab.iloc[tr][FEATS], y[tr])
    oof[va] = m.predict_proba(lab.iloc[va][FEATS])[:, 1]
print('OOF ROC-AUC %.4f' % roc_auc_score(y, oof))
imp = pd.Series(m.booster_.feature_importance('gain'), index=FEATS)
display((imp / imp.sum()).sort_values(ascending=False).head(8).round(3).to_frame('gain share'))

OOF ROC-AUC 0.8906


,gain share
crosshair_placement,0.228
reports_received,0.214
account_age_days,0.172
friend_network_size,0.158
level,0.071
game_sense_score,0.026
headshot_percentage,0.022
kill_death_ratio,0.020


LightGBM handles missing values natively. Heavier regularisation (small trees, many slow steps, row/column subsampling) was slightly better than the default on the test-like mix.

## 5. Threshold for binary F2

In [6]:
GRID = np.linspace(0.05, 0.5, 181)
curve = pd.DataFrame({'threshold': GRID,
                      'binary F2': [f2(y, (oof >= t).astype(int), W) for t in GRID],
                      'macro F2': [macro_f2(y, (oof >= t).astype(int), W) for t in GRID]})
THR = float(curve.loc[curve['binary F2'].idxmax(), 'threshold'])
t_macro = float(curve.loc[curve['macro F2'].idxmax(), 'threshold'])
rows = []
for name, t in [('tuned for macro F2', t_macro), ('tuned for binary F2 (used)', THR)]:
    p = (oof >= t).astype(int)
    rows.append({'threshold': round(t, 3), 'flags as cheater': f'{np.average(p, weights=W):.1%}',
                 'binary F2 (Kaggle)': round(f2(y, p, W), 4), 'macro F2': round(macro_f2(y, p, W), 4), 'choice': name})
display(pd.DataFrame(rows).set_index('choice'))

# honest check: threshold chosen on 4 folds, applied to the 5th
pred = np.zeros(len(y), int)
for tr, va in folds:
    s = [f2(y[tr], (oof[tr] >= t).astype(int), W[tr]) for t in GRID]
    pred[va] = (oof[va] >= GRID[int(np.argmax(s))]).astype(int)
print(f'chosen threshold {THR:.3f} | nested binary F2 (test-weighted) {f2(y, pred, W):.4f}')
print('theory check (calibrated probabilities): best threshold ≈ best F2 / 5 = %.3f' % (curve['binary F2'].max() / 5))

,threshold,flags as cheater,binary F2 (Kaggle),macro F2
choice,,,,
tuned for macro F2,0.393,40.1%,0.7768,0.8019
tuned for binary F2 (used),0.205,60.0%,0.8501,0.7504


chosen threshold 0.205 | nested binary F2 (test-weighted) 0.8497
theory check (calibrated probabilities): best threshold ≈ best F2 / 5 = 0.170


- The binary-F2 threshold (≈0.2) flags about 60% of players, against about 40% for the macro-F2 threshold.
  On the leaderboard this change raised Task 1 from ≈0.78 to ≈0.85.
- The curve is flat around the optimum (±0.03 changes binary F2 by < 0.001), so the exact value is not fragile.
- **Business meaning:** the review team sees more false alarms, but misses far fewer real cheaters, which is what a pre-filter is for.

## 6. What else was tried (test-weighted score, same folds)

| Idea | Result vs baseline LightGBM |
|---|---|
| Regularised LightGBM (kept) | +0.001 |
| Model-imputed key features (from correlated columns) | +0.001 |
| Average of 3 models (LightGBM ×2 + CatBoost) | +0.001 |
| Missing-count feature | 0 |
| Pseudo-labelling the 13,184 unlabelled rows | −0.000 |
| Ratio / consistency features (level vs age × speed, reports per level, …) | 0 |
| Sample weights matching the test mix | −0.002 |
| CatBoost alone | −0.004 |
| Training only on complete rows | −0.039 |

Other checks: no gap-separated pure groups (only `crosshair_placement` has pure tails, which trees already use); no leaks in
`player_id`, row order or missingness; probabilities are well calibrated. Everything lands within ±0.001, so the remaining
error is overlap between cheaters and legit players that these features cannot separate.

## 7. Final model on all labelled rows → test predictions

In [7]:
final = lgb.LGBMClassifier(**PARAMS).fit(lab[FEATS], y)
p_test = final.predict_proba(test[FEATS])[:, 1]
sample = pd.read_csv(SAMPLE)
sub = pd.DataFrame({'id': sample['id'], 'task1': (p_test >= THR).astype(int)})   # row i of test.csv = row i of the sample file
assert len(sub) == len(test) == 25889 and sub['task1'].isin([0, 1]).all()
sub.to_csv('task1_prediction.csv', index=False)
print(f'threshold {THR:.3f} | flagged as cheater on test: {sub.task1.mean():.1%} | saved task1_prediction.csv')
sub.head()

threshold 0.205 | flagged as cheater on test: 59.9% | saved task1_prediction.csv


,id,task1
0,ANS00001,1
1,ANS00002,1
2,ANS00003,1
3,ANS00004,1
4,ANS00005,0
